In [3]:
import asyncio
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

True

Your Jupyter Notebook (.ipynb) cell runs perfectly without explicitly using the asyncio library here because of how Jupyter is designed.
There are two separate reasons why this works in your specific case:
### 1. Jupyter has a "Built-In" Event Loop Engine
Jupyter Notebooks run on a background process called IPython (ipykernel). IPython automatically starts and manages a global asyncio event loop for you behind the scenes.
Because the loop is always running, Jupyter supports a special feature called autoawait. It automatically intercepts any await keyword written at the "top level" of a notebook cell, wraps your code under the hood, and executes it inside that hidden event loop. You do not need to call asyncio.run(), which means you don't need to touch the asyncio library to run it.
### 2. You are using an External Async Client
The client object in your cell is an instance of `MultiServerMCPClient` from the `langchain_mcp_adapters` library.

* The Authors Did the Work: The developers who wrote `MultiServerMCPClient` defined `get_tools()`, `get_resources()`, and `get_prompt()` as asynchronous functions (coroutines).
* Native Keywords: Because await is a native piece of Python syntax, Python recognizes it perfectly fine without any imports.

### A Quick Paradox in Your Code
If you look closely at your notebook's very first cell, you actually did import asyncio alongside sys and dotenv:

```python
import asyncioimport sys
from dotenv import load_dotenv
...
```

Even though you imported it there, you'll notice that you commented out the block in the next cell that actually uses it (the Windows Proactor loop fix). If you completely delete import asyncio from that first cell, your await client.get_tools() cell will still work perfectly fine for the exact reasons above.

In [1]:
# # Fix for Windows issues in Jupyter notebooks
# if sys.platform == "win32":
#     # 1. Use ProactorEventLoop for subprocess support
#     if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
#         asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

#     # 2. Redirect stderr to avoid fileno() error when launching MCP servers
#     if "ipykernel" in sys.modules:
#         sys.stderr = sys.__stderr__

## Local MCP server from ./resources/2.1_mcp_server.py

In [ ]:
client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
    )

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [7]:
print(prompt)


    You are a helpful assistant that answers user questions about LangChain, LangGraph and LangSmith.

    You can use the following tools/resources to answer user questions:
    - search_web: Search the web for information
    - github_file: Access the langchain-ai repo files

    If the user asks a question that is not related to LangChain, LangGraph or LangSmith, you should say 
    "I'm sorry, I can only answer questions about LangChain, LangGraph and LangSmith."

    You may try multiple tool and resource calls to answer the user's question.

    You may also ask clarifying questions to the user to better understand their question.
    


In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=str(prompt)
    )

In [ ]:
config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config # type: ignore
    ) 

In [ ]:
from rich.pretty import pprint as rpprint

rpprint(response, indent_guides=True, expand_all=True)

{
│   'messages': [
│   │   HumanMessage(
│   │   │   content='Tell me about the langchain-mcp-adapters library',
│   │   │   additional_kwargs={},
│   │   │   response_metadata={},
│   │   │   id='2ac096b1-836e-4975-b9f5-3cffe0cd727b'
│   │   ),
│   │   AIMessage(
│   │   │   content='',
│   │   │   additional_kwargs={
│   │   │   │   'refusal': None
│   │   │   },
│   │   │   response_metadata={
│   │   │   │   'token_usage': {
│   │   │   │   │   'completion_tokens': 93,
│   │   │   │   │   'prompt_tokens': 273,
│   │   │   │   │   'total_tokens': 366,
│   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'reasoning_tokens': 64,
│   │   │   │   │   │   'rejected_prediction_tokens': 0
│   │   │   │   │   },
│   │   │   │   │   'prompt_tokens_details': {
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'cached_tokens': 0
│   │   │   │   │   }
│   │   │   │   },
│   │   │   │   'model_provider': 'openai',
│   │   │   │   'model_name': 'gpt-5-nano-2025-08-07',
│   │   │   │   'system_fingerprint': None,
│   │   │   │   'id': 'chatcmpl-EPT7LxXy5IW2dreZYZHlP0KU2z8rU',
│   │   │   │   'service_tier': 'default',
│   │   │   │   'finish_reason': 'tool_calls',
│   │   │   │   'logprobs': None
│   │   │   },
│   │   │   id='lc_run--01a0b4bf-5535-7572-8608-fe340b5ca232-0',
│   │   │   tool_calls=[
│   │   │   │   {
│   │   │   │   │   'name': 'search_web',
│   │   │   │   │   'args': {
│   │   │   │   │   │   'query': 'langchain-mcp-adapters library'
│   │   │   │   │   },
│   │   │   │   │   'id': 'call_OXXnAU2i8IUArUhpwtagG9J0',
│   │   │   │   │   'type': 'tool_call'
│   │   │   │   }
│   │   │   ],
│   │   │   invalid_tool_calls=[],
│   │   │   usage_metadata={
│   │   │   │   'input_tokens': 273,
│   │   │   │   'output_tokens': 93,
│   │   │   │   'total_tokens': 366,
│   │   │   │   'input_token_details': {
│   │   │   │   │   'audio': 0,
│   │   │   │   │   'cache_read': 0
│   │   │   │   },
│   │   │   │   'output_token_details': {
│   │   │   │   │   'audio': 0,
│   │   │   │   │   'reasoning': 64
│   │   │   │   }
│   │   │   }
│   │   ),
│   │   ToolMessage(
│   │   │   content=[
│   │   │   │   {
│   │   │   │   │   'type': 'text',
│   │   │   │   │   'text': '{\n  "query": "langchain-mcp-adapters library",\n  "follow_up_questions": null,\n  "answer": null,\n  "images": [],\n  "results": [\n    {\n      "url": "https://medium.com/@deepakkamboj/the-complete-guide-to-langchain-mcp-adapters-bridging-langchain-and-model-context-protocol-3f5507cbd3ca",\n      "title": "The Complete Guide to langchain-mcp-adapters - Medium",\n      "content": "## Conclusion\\n\\nThe `langchain-mcp-adapters` library represents a crucial piece of infrastructure for modern AI agent development. By bridging LangChain\'s powerful agent framework with MCP\'s standardized tool ecosystem, it enables developers to build sophisticated, multi-tool applications without the burden of custom integrations. [...] ### Core Purpose\\n\\nThe `langchain-mcp-adapters` library serves three primary functions:\\n\\n1.   Tool Conversion: Automatically converts MCP tools into LangChain and LangGraph-compatible tools\\n2.   Multi-Server Management: Enables simultaneous interaction with multiple MCP servers\\n3.   Seamless Integration: Provides a unified interface without requiring manual MCP client management\\n\\n### Key Benefits [...] As the MCP ecosystem continues to mature with backing from major AI companies and the Linux Foundation, `langchain-mcp-adapters` positions developers to leverage this growing standard effectively. Whether building enterprise automation, development tools, or customer-facing AI applications, this library provides the foundation for scalable, maintainable tool integration.\\n\\n## Additional Resources",\n      "score": 0.9274752,\n      "raw_content": null,\n      "id": "efd18c-00"\n    },\n    {\n      "

In [11]:
import textwrap

text = response["messages"][-1].content
wrapped = "\n\n".join(textwrap.fill(p, width=110) for p in text.split("\n\n"))
print(wrapped)

Here’s a concise overview of the LangChain MCP Adapters library and how it fits into the LangChain ecosystem.

What it is - A library that makes Anthropic Model Context Protocol (MCP) tools compatible with LangChain and
LangGraph. - It acts as a bridge between MCP servers (which expose tools) and LangChain agents, so you can use
MCP tools inside LangChain workflows without writing custom adapters.

What problems it solves - Tool interoperability: MCP tools can be wrapped as LangChain tools automatically. -
Multi-server access: You can connect to and manage tools from multiple MCP servers in one place. - Unified
tooling interface: You get a consistent LangChain API for discovery, invocation, and management of MCP tools.

Core ideas and components - MCP is the Model Context Protocol: an open standard for how tools and model
context are provided to language models. - The adapter layer translates MCP tools into LangChain-compatible
tools that a LangChain/LangGraph agent can invoke. - Suppo

In [14]:
print(response["messages"][-1].content)

Here’s a concise overview of the LangChain MCP Adapters library and how it fits into the LangChain ecosystem.

What it is
- A library that makes Anthropic Model Context Protocol (MCP) tools compatible with LangChain and LangGraph.
- It acts as a bridge between MCP servers (which expose tools) and LangChain agents, so you can use MCP tools inside LangChain workflows without writing custom adapters.

What problems it solves
- Tool interoperability: MCP tools can be wrapped as LangChain tools automatically.
- Multi-server access: You can connect to and manage tools from multiple MCP servers in one place.
- Unified tooling interface: You get a consistent LangChain API for discovery, invocation, and management of MCP tools.

Core ideas and components
- MCP is the Model Context Protocol: an open standard for how tools and model context are provided to language models.
- The adapter layer translates MCP tools into LangChain-compatible tools that a LangChain/LangGraph agent can invoke.
- Suppo

## Online MCP

In [4]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "--with", "mcp<2",  # pin uvx to install with an older, compatible mcp alongside it
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [5]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [6]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

In [7]:
from rich.pretty import pprint as rpprint

rpprint(response, indent_guides=True, expand_all=True)

{
│   'messages': [
│   │   HumanMessage(
│   │   │   content='What time is it?',
│   │   │   additional_kwargs={},
│   │   │   response_metadata={},
│   │   │   id='886a996c-aaf2-4d5f-83cc-231adce9f1f3'
│   │   ),
│   │   AIMessage(
│   │   │   content='',
│   │   │   additional_kwargs={
│   │   │   │   'refusal': None
│   │   │   },
│   │   │   response_metadata={
│   │   │   │   'token_usage': {
│   │   │   │   │   'completion_tokens': 155,
│   │   │   │   │   'prompt_tokens': 295,
│   │   │   │   │   'total_tokens': 450,
│   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'reasoning_tokens': 128,
│   │   │   │   │   │   'rejected_prediction_tokens': 0
│   │   │   │   │   },
│   │   │   │   │   'prompt_tokens_details': {
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'cached_tokens': 0
│   │   │   │   │   }
│   │   │   │   },
│   │   │   │   'model_provider': 'openai',
│   │   │   │   'model_name': 'gpt-5-nano-2025-08-07',
│   │   │   │   'system_fingerprint': None,
│   │   │   │   'id': 'chatcmpl-EPThclqKizJAPUxuVUOz7AZgoTS1n',
│   │   │   │   'service_tier': 'default',
│   │   │   │   'finish_reason': 'tool_calls',
│   │   │   │   'logprobs': None
│   │   │   },
│   │   │   id='lc_run--01a0b4e1-a3d1-7d50-8c0b-1c624873bd91-0',
│   │   │   tool_calls=[
│   │   │   │   {
│   │   │   │   │   'name': 'get_current_time',
│   │   │   │   │   'args': {
│   │   │   │   │   │   'timezone': 'America/New_York'
│   │   │   │   │   },
│   │   │   │   │   'id': 'call_eJtNjBRdmJgo2T76s4JLFT1u',
│   │   │   │   │   'type': 'tool_call'
│   │   │   │   }
│   │   │   ],
│   │   │   invalid_tool_calls=[],
│   │   │   usage_metadata={
│   │   │   │   'input_tokens': 295,
│   │   │   │   'output_tokens': 155,
│   │   │   │   'total_tokens': 450,
│   │   │   │   'input_token_details': {
│   │   │   │   │   'audio': 0,
│   │   │   │   │   'cache_read': 0
│   │   │   │   },
│   │   │   │   'output_token_details': {
│   │   │   │   │   'audio': 0,
│   │   │   │   │   'reasoning': 128
│   │   │   │   }
│   │   │   }
│   │   ),
│   │   ToolMessage(
│   │   │   content=[
│   │   │   │   {
│   │   │   │   │   'type': 'text',
│   │   │   │   │   'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-09-18T10:18:06-04:00",\n  "day_of_week": "Friday",\n  "is_dst": true\n}',
│   │   │   │   │   'id': 'lc_45dcaa84-cfee-4aa7-b898-f41f7bdde775'
│   │   │   │   }
│   │   │   ],
│   │   │   name='get_current_time',
│   │   │   id='d528ae5c-d6cf-44ae-8934-77c6d7d58a5d',
│   │   │   tool_call_id='call_eJtNjBRdmJgo2T76s4JLFT1u'
│   │   ),
│   │   AIMessage(
│   │   │   content='It’s 10:18:06 AM on Friday, September 18, 2026 in New York (EDT, UTC-4). Want me to convert this to another time zone?',
│   │   │   additional_kwargs={
│   │   │   │   'refusal': None
│   │   │   },
│   │   │   response_metadata={
│   │   │   │   'token_usage': {
│   │   │   │   │   'completion_tokens': 497,
│   │   │   │   │   'prompt_tokens': 378,
│   │   │   │   │   'total_tokens': 875,
│   │   │   │   │   'completion_tokens_details': {
│   │   │   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'reasoning_tokens': 448,
│   │   │   │   │   │   'rejected_prediction_tokens': 0
│   │   │   │   │   },
│   │   │   │   │   'prompt_tokens_details': {
│   │   │   │   │   │   'audio_tokens': 0,
│   │   │   │   │   │   'cached_tokens': 0
│   │   │   │   │   }
│   │   │   │   },
│   │   │   │   'model_provider': 'openai',
│   │   │   │   'model_name': 'gpt-5-nano-2025-08-07',
│   │   │   │   'system_fingerprint': None,
│   │   │   │   'id': 'chatcmpl-EPThfMK3sBKAGpMJV9jSbJD5Zqiqs',
│   │   │   │   'service_tier': 'default',
│   │   │   │   'finish_reason': 'stop',
│   │   │   │   'logprobs': None
│   │   │   },
│   │   │   id='lc_run--01a0b4e1-b7ab-7501-a31d-4323b534aebd-0',
│   │  

In [9]:
print(response["messages"][-1].content)

It’s 10:18:06 AM on Friday, September 18, 2026 in New York (EDT, UTC-4). Want me to convert this to another time zone?
